In [ ]:
"""Inserts"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Any, Callable


### Test

In [ ]:
"""Reading data"""
training_data=pd.read_parquet("data_parquet_2026/train.parquet")
sensors = pd.read_parquet("data_parquet_2026/sensors.parquet")
test_data = pd.read_parquet("data_parquet_2026/test.parquet")

In [ ]:
""" Ségrégation des profiles"""
n=int(len(training_data["power"])/len(training_data["sensor"].unique())/3) #27384/3 = 9128
power1 = training_data.query("sensor =='N0000'")
power2 = training_data.query("sensor =='N0000'")
power3 = training_data.query("sensor =='N0000'")

for senso in training_data["sensor"].unique():
    power1 = pd.concat((power1, training_data[training_data["sensor"] == senso][:n]))
    power2 = pd.concat((power2, training_data[training_data["sensor"] == senso][n+1:2*n]))
    power3 = pd.concat((power3, training_data[training_data["sensor"] == senso][2*n+1:]))

#power1.plot.scatter(x="time", y="power", alpha=0.5)
#power2.plot.scatter(x="time", y="power", alpha=0.5)
#power3.plot.scatter(x="time", y="power", alpha=0.5)

In [ ]:
training_data.info()
#training_data.head(5)
training_data["sensor"]
sensors["sensor"]
#test_data.sample(15)
#sensors.info()
#sensors.query("coor_z == 0") #tous =0
#sensors.head(21)




In [ ]:
#for sens in training_data["sensors"]:
 #   print(sens)
sampleP=training_data.query("time <= 1800000 and sensor=='N418'")
sampleP.head(10)

sampleP.plot.scatter(x="time", y="temperature", alpha=0.5)
#mean.plot.scatter(x="time", y="temperature", alpha=0.5)

In [ ]:
"""graphes tous pour 1 senseur"""

sampleN418 = power1.query(" sensor=='N418'")
#sampleN418.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418.plot.scatter(x="time", y="temperature", alpha=0.5)
sensors.query("sensor == 'N418'")
sampleN418.head()

In [ ]:
b = training_data.query("time <= 10**9 and time >= 0.8*10**9")
b.plot.scatter(x="time", y="power", alpha=0.5)
b.plot.scatter(x="time", y="temperature", alpha=0.5)



In [ ]:
#for sens in training_data["sensors"]:
 #   print(sens)
sampleP=training_data.query("time <= 1800000 and sensor=='N418'")
sampleP.head(10)

sampleP.plot.scatter(x="power", y="temperature", alpha=0.5)
#mean.plot.scatter(x="time", y="temperature", alpha=0.5)

In [ ]:
a = sampleP["power"].diff(periods=864000)

a.head(20)
power1 = training_data.query("power == 500.0000 or power == 1000.0000 or power == 1500.0000")
power1.plot.scatter(x="time", y="power", alpha=0.5)


In [ ]:
"""graphes tous pour 1 senseur"""

sampleN418 = training_data.query(" sensor=='N418'")
sampleN418.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418.plot.scatter(x="power", y="temperature", alpha=0.5)
sensors.query("sensor == 'N418'")

In [ ]:
b = training_data.query("time <= 10**9 and time >= 0.8*10**9")
b.plot.scatter(x="time", y="power", alpha=0.5)
b.plot.scatter(x="time", y="temperature", alpha=0.5)
c=b["power"].pct_change(periods=1)


In [ ]:
sampleN418b = sampleN418.sample(1000)
sampleN418b.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418b.plot.scatter(x="time", y="temperature", alpha=0.5)


## I. First Model: KNN Method

### 1. Distance Metrics

In [ ]:
def man_dist(sample_coo: np.ndarray, X: np.ndarray) -> np.ndarray:

    """Computes the Manhattan distance between a sample and the sensors.
    Args:
        sample_coo: Coordinates of the sample, of shape (D, )
        X: Dataset of the sensors' coordinates, of shape (N, D)
    Returns:
        dist: Distances of shape (N,)
    """

    return np.abs(X - sample_coo).sum(axis = 1)
   


def eucli_dist(sample_coo: np.ndarray, X: np.ndarray) -> np.ndarray:

    """Computes the Euclidean distance between a sample and the sensors.
    Args:
        sample_coo: Coordinates of the sample, of shape (D, )
        X: Dataset of the sensors' coordinates, of shape (N, D)
    Returns:
        dist: Distances of shape (N,)
    """ 
   
    return np.sqrt(((X - sample_coo) ** 2).sum(axis = 1))

### 2. Distance Score

In [ ]:
def distance_score(distances:np.ndarray,
                   score_parameter: float = 1) -> np.ndarray:
    
    """Gives a score to each given sensor, depending on their distance to sample.
    The closer is the sample, the higher the score. We set an upper limit to it
    to avoid giving "blind confidence" to "very close" sensors.
    Args:
        distances: distances calculated of each sensor to the sample
        score_parameter: Hyper-parameter, set to 1 by default
    Returns:
        scores: Distance' scores of shape (N,)
    """

    scores = 1 / (score_parameter * distances + 1)
    
    return scores


### 3. Missing Values manager

In [ ]:
def managing_missing_val(sens : pd.DataFrame):
    
    """
    """
    
    sens = sens.reset_index(drop=True)
    temp = 0
    memo = []

    for i, val in enumerate(sens["temperature"]):
        
        if not val == val: # if nan
            memo.append(i)       
                     
        else: # c'est pas un nan
            if temp is not int: # si on a une valeur en mémoire
                if len(memo) != 0: # et qu'il y avait des nan avant
                    sens.loc[memo,"temperature"] = (temp+sens["temperature"][i]) / 2 # on remplace les nan par la moyenne de la mémoire et l'actuel
                    memo = [] # on oublie
                temp = sens["temperature"][i]
                
            else: # pas de valeur en mémoire
                if len(memo) != 0: # mais des nan avant
                    sens.loc[memo,"temperature"]=sens["temperature"][i] # on remplaces les nan par la valeur d'après
                    memo = [] # on oublie
        
    return sens

### 4. K-Nearest Neighbors

In [ ]:
def find_nearest_neighbors(
    sample_coo: np.ndarray, 
    dataset: np.ndarray, 
    distance_fn: Callable = eucli_dist, 
    k: int = 7) -> np.ndarray:
    
    """Finds the names and distances of the k-Nearest Neighbors of one sample.
    Args:
        sample_coo: Coordinates of the sample, of shape (D, )
        dataset: Dataset of the the sensors' coordinates, of shape (N, D)
        distance_fn: Distance function
        k: Number of nearest neighbors taken, set to 7 by default

    Returns:
        An array of shape (k, 2), with:
            neighbors_names: sensors' names, of shape (1, k)
            neighbors_distances: Distances of each neighbor to the sample, of shape (1, k)
    """

    distances = distance_fn(sample_coo, dataset)
    
    neighbors_indices = (distances)[:k]
    neighbors_distances = distances[neighbors_indices]
    neighbors_names = ???
    
    return np.array([neighbors_names, neighbors_distances]).T

### 5. Prediction Functions

In [ ]:
def prediction_single(
    sample_coo: np.ndarray, 
    dataset_coo: np.ndarray,
    dataset_temp_at_t:np.ndarray,
    distance_fn: Callable = eucli_dist, 
    k: int = 7) -> float:

    """Gives the final prediction of a given sample's temperature, while calling KNN. Calculates the
    mean value of the temperatures of the neighbors, weighted by their distance scores.
    Args:
        sample_coo: Coordinates of the sample, of shape (D, )
        dataset_coo: Dataset of the sensors' coordinates, of shape (N, D)
        dataset_temp_at_t: Dataset of the sensors' temperatures at the time of the sample, of shape (N, D)
        distance_fn: Distance function
        k: Number of nearest neighbors taken, set to 7 by default
    Returns:
        pred: The value of the predicted temperature of the sample
    """

    KNN_returns = find_nearest_neighbors(sample_coo = sample_coo, dataset = dataset_coo, distance_fn = distance_fn, k = k)
    neighbors = KNN_returns[0]
    temperatures = 
    distances = KNN_returns[1]
    scores = distance_score(distances)
    pred = np.sum(scores * temperatures) / np.sum(scores)

    return pred



def prediction(
    samples_coo: np.ndarray, 
    dataset_coo: np.ndarray,
    dataset_temp_at_t:np.ndarray,
    distance_fn: Callable = eucli_dist, 
    k: int = 7) -> float:

    """Gives the final prediction of a given sample's temperature, while calling KNN. Calculates the
    mean value of the temperatures of the neighbors, weighted by their distance scores.
    Args:
        samples_coo: Coordinates of all samples, of shape (M, D)
        dataset_coo: Dataset of the sensors' coordinates, of shape (N, D)
        dataset_temp_at_t: Dataset of the sensors' temperatures at the time of the sample, of shape (N, D)
        distance_fn: Distance function
        k: Number of nearest neighbors taken, set to 7 by default
    Returns:
        pred: All values of the samples' predicted temperatures
    """


    

### 6. Loss Function

### Application

In [ ]:
# Reading data
training_data=pd.read_parquet("data_parquet_2026/train.parquet")
sensors = pd.read_parquet("data_parquet_2026/sensors.parquet").drop(columns=["coor_z"])
test_data = pd.read_parquet("data_parquet_2026/test.parquet")

In [ ]:
# Partition of the dataset into "training" and "validation" sets
validation_sensors_coo = sensors.sample(frac = 1/5).sort_index()
train_sensors_coo = sensors.drop(validation_sensors_coo.index)

# nearest_sensors = sensors["sensor"][find_nearest_neighbors(..., )] # A compléter avec sample_coo

## II. A few improvements

### 1. Outliers Manager

### 2. n-Folder Validation


### Application

## III. Second model: Linear Regression

### 1. Prediction Function

### 2. Penalized Loss Function

### 3. Gradient Descent

### Application

## Tests post-fonctions

In [ ]:
sampleN418.isna().sum()
a=managing_missing_val(sampleN418)
a.isna().sum()